In [89]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
from datasets import load_dataset
import matplotlib.pyplot as plt

# ================================
# 1. Residual embedding denoiser (predicts noise)
# ================================
class ResidualDenoiser(nn.Module):
    def __init__(self, embedding_dim=512, hidden_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )

    def forward(self, x_noisy, x_clean):
        # predict residual noise
        residual = self.net(x_noisy - x_clean)
        return x_noisy - residual  # remove predicted noise

# ================================
# 2. Helper: add noise
# ================================
def add_noise(embeddings, noise_std=0.2):
    return embeddings + torch.randn_like(embeddings) * noise_std

# ================================
# 3. Load dataset
# ================================
dataset = load_dataset("SKyu/my-image-captioning-dataset")
train_dataset = dataset["train"].select(range(2000, 3000))
validation_dataset = dataset["train"].select(range(1, 1000))

def transformFn(sample):
    images = [item["image"] for item in sample]
    captions = [item["prompt"] for item in sample]
    return images, captions

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=transformFn)
val_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, collate_fn=transformFn)

# ================================
# 4. Load frozen CLIP
# ================================
device = "cpu"
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = AutoModelForZeroShotImageClassification.from_pretrained("openai/clip-vit-base-patch32")
model.to(device)
model.eval()
for param in model.parameters():
    param.requires_grad = False

# ================================
# 5. Initialize residual denoiser
# ================================
denoiser = ResidualDenoiser(embedding_dim=512, hidden_dim=1024).to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# ================================
# 6. Train denoiser (residual)
# ================================
num_epochs = 10
for epoch in range(num_epochs):
    denoiser.train()
    epoch_loss = 0
    for images, captions in train_loader:
        inputs = processor(text=captions, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=77).to(device)

        with torch.no_grad():
            image_embeds = F.normalize(model.get_image_features(inputs["pixel_values"]), dim=-1)
            text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
            text_embeds = F.normalize(model.get_text_features(**text_inputs), dim=-1)

        # Add noise
        noisy_image = add_noise(image_embeds)
        noisy_text = add_noise(text_embeds)

        # Residual denoiser
        denoised_image = denoiser(noisy_image, image_embeds)
        denoised_text = denoiser(noisy_text, text_embeds)

        # Loss: MSE between denoised and clean embeddings
        loss = criterion(denoised_image, image_embeds) + criterion(denoised_text, text_embeds)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader):.6f}")

print("✅ Residual denoiser training complete!")

# ================================
# 7. Functions to get embeddings
# ================================
def get_raw_embeddings(images, captions):
    model.eval()
    with torch.no_grad():
        inputs = processor(text=captions, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=77).to(device)
        image_embeds = F.normalize(model.get_image_features(inputs["pixel_values"]), dim=-1)
        text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
        text_embeds = F.normalize(model.get_text_features(**text_inputs), dim=-1)
    return image_embeds, text_embeds

def get_denoised_embeddings(images, captions):
    denoiser.eval()
    image_embeds, text_embeds = get_raw_embeddings(images, captions)
    denoised_image = F.normalize(denoiser(image_embeds, image_embeds), dim=-1)
    denoised_text = F.normalize(denoiser(text_embeds, text_embeds), dim=-1)
    return denoised_image, denoised_text

# ================================
# 8. Evaluation
# ================================
def evaluate(loader, use_denoiser=False):
    recall_at_1 = 0
    recall_at_3 = 0
    recall_at_5 = 0
    total = 0

    for images, captions in loader:
        if use_denoiser:
            image_embeds, text_embeds = get_denoised_embeddings(images, captions)
        else:
            image_embeds, text_embeds = get_raw_embeddings(images, captions)

        sim_matrix = torch.matmul(image_embeds, text_embeds.T)
        for i in range(sim_matrix.size(0)):
            scores = sim_matrix[i]
            top1 = scores.topk(min(1, scores.size(0))).indices[0].item()
            top3 = scores.topk(min(3, scores.size(0))).indices.tolist()
            top5 = scores.topk(min(5, scores.size(0))).indices.tolist()
            true_id = i
            if top1 == true_id:
                recall_at_1 += 1
            if true_id in top3:
                recall_at_3 += 1
            if true_id in top5:
                recall_at_5 += 1
            total += 1

    return {
        "Recall@1": recall_at_1 / total * 100,
        "Recall@3": recall_at_3 / total * 100,
        "Recall@5": recall_at_5 / total * 100
    }

# ================================
# 9. Compare raw vs residual denoised embeddings
# ================================
raw_recalls = evaluate(val_loader, use_denoiser=False)
denoised_recalls = evaluate(val_loader, use_denoiser=True)

print("📌 CLIP embeddings (raw):", raw_recalls)
print("📌 Residual denoised embeddings:", denoised_recalls)


Epoch 1/10, Loss: 0.082361
Epoch 2/10, Loss: 0.079632
Epoch 3/10, Loss: 0.077116
Epoch 4/10, Loss: 0.074777
Epoch 5/10, Loss: 0.072524
Epoch 6/10, Loss: 0.070110
Epoch 7/10, Loss: 0.067944
Epoch 8/10, Loss: 0.065555
Epoch 9/10, Loss: 0.063035
Epoch 10/10, Loss: 0.060692
✅ Residual denoiser training complete!
📌 CLIP embeddings (raw): {'Recall@1': 40.94094094094094, 'Recall@3': 66.46646646646647, 'Recall@5': 76.87687687687688}
📌 Residual denoised embeddings: {'Recall@1': 38.43843843843844, 'Recall@3': 64.46446446446447, 'Recall@5': 74.87487487487488}


In [90]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
from datasets import load_dataset
import matplotlib.pyplot as plt

# ================================
# 1. Residual denoiser (predicts noise)
# ================================
class ResidualDenoiser(nn.Module):
    def __init__(self, embedding_dim=512, hidden_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )

    def forward(self, residual):
        return self.net(residual)

# ================================
# 2. Helper: add noise
# ================================
def add_noise(embeddings, noise_std=0.2):
    return embeddings + torch.randn_like(embeddings) * noise_std

# ================================
# 3. Load dataset
# ================================
dataset = load_dataset("SKyu/my-image-captioning-dataset")
train_dataset = dataset["train"].select(range(1000, 2000))
validation_dataset = dataset["train"].select(range(2000, 3000))

def transformFn(sample):
    images = [item["image"] for item in sample]
    captions = [item["prompt"] for item in sample]
    return images, captions

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=transformFn)
val_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, collate_fn=transformFn)

# ================================
# 4. Load frozen CLIP
# ================================
device = "cpu"
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = AutoModelForZeroShotImageClassification.from_pretrained("openai/clip-vit-base-patch32")
model.to(device)
model.eval()
for param in model.parameters():
    param.requires_grad = False

# ================================
# 5. Initialize residual denoiser
# ================================
denoiser = ResidualDenoiser(embedding_dim=512, hidden_dim=1024).to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# ================================
# 6. Train denoiser with clean + noisy mix
# ================================
num_epochs = 10

for epoch in range(num_epochs):
    denoiser.train()
    epoch_loss = 0
    for images, captions in train_loader:
        inputs = processor(text=captions, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=77).to(device)

        with torch.no_grad():
            image_embeds = F.normalize(model.get_image_features(inputs["pixel_values"]), dim=-1)
            text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
            text_embeds = F.normalize(model.get_text_features(**text_inputs), dim=-1)

        # Mix of clean and noisy embeddings
        if torch.rand(1) < 0.5:
            noisy_image = add_noise(image_embeds)
            noisy_text = add_noise(text_embeds)
        else:
            noisy_image = image_embeds.clone()
            noisy_text = text_embeds.clone()

        # Residual denoiser: predict noise
        denoised_image = noisy_image - denoiser(noisy_image - image_embeds)
        denoised_text = noisy_text - denoiser(noisy_text - text_embeds)

        # Loss
        loss = criterion(denoised_image, image_embeds) + criterion(denoised_text, text_embeds)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader):.6f}")

print("✅ Residual denoiser (with clean+noisy mix) training complete!")

# ================================
# 7. Functions to get embeddings
# ================================
def get_raw_embeddings(images, captions):
    model.eval()
    with torch.no_grad():
        inputs = processor(text=captions, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=77).to(device)
        image_embeds = F.normalize(model.get_image_features(inputs["pixel_values"]), dim=-1)
        text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
        text_embeds = F.normalize(model.get_text_features(**text_inputs), dim=-1)
    return image_embeds, text_embeds

def get_denoised_embeddings(images, captions):
    denoiser.eval()
    image_embeds, text_embeds = get_raw_embeddings(images, captions)
    denoised_image = F.normalize(image_embeds - denoiser(torch.zeros_like(image_embeds)), dim=-1)
    denoised_text = F.normalize(text_embeds - denoiser(torch.zeros_like(text_embeds)), dim=-1)
    return denoised_image, denoised_text

# ================================
# 8. Evaluation
# ================================
def evaluate(loader, use_denoiser=False):
    recall_at_1 = 0
    recall_at_3 = 0
    recall_at_5 = 0
    total = 0

    for images, captions in loader:
        if use_denoiser:
            image_embeds, text_embeds = get_denoised_embeddings(images, captions)
        else:
            image_embeds, text_embeds = get_raw_embeddings(images, captions)

        sim_matrix = torch.matmul(image_embeds, text_embeds.T)
        for i in range(sim_matrix.size(0)):
            scores = sim_matrix[i]
            top1 = scores.topk(min(1, scores.size(0))).indices[0].item()
            top3 = scores.topk(min(3, scores.size(0))).indices.tolist()
            top5 = scores.topk(min(5, scores.size(0))).indices.tolist()
            true_id = i
            if top1 == true_id:
                recall_at_1 += 1
            if true_id in top3:
                recall_at_3 += 1
            if true_id in top5:
                recall_at_5 += 1
            total += 1

    return {
        "Recall@1": recall_at_1 / total * 100,
        "Recall@3": recall_at_3 / total * 100,
        "Recall@5": recall_at_5 / total * 100
    }

# ================================
# 9. Compare raw vs residual denoised embeddings
# ================================
raw_recalls = evaluate(val_loader, use_denoiser=False)
denoised_recalls = evaluate(val_loader, use_denoiser=True)

print("📌 CLIP embeddings (raw):", raw_recalls)
print("📌 Residual denoised embeddings:", denoised_recalls)


Epoch 1/10, Loss: 0.044334
Epoch 2/10, Loss: 0.060565
Epoch 3/10, Loss: 0.022271
Epoch 4/10, Loss: 0.024404
Epoch 5/10, Loss: 0.040670
Epoch 6/10, Loss: 0.051116
Epoch 7/10, Loss: 0.031802
Epoch 8/10, Loss: 0.048668
Epoch 9/10, Loss: 0.028113
Epoch 10/10, Loss: 0.027432
✅ Residual denoiser (with clean+noisy mix) training complete!
📌 CLIP embeddings (raw): {'Recall@1': 39.900000000000006, 'Recall@3': 66.7, 'Recall@5': 77.5}
📌 Residual denoised embeddings: {'Recall@1': 40.1, 'Recall@3': 66.8, 'Recall@5': 77.7}
